# Lab 05-01 — Query rewrite: fix the query before you embed it

**Track 05 · Query transformation** — retrieval quality is capped by query quality: a vague or conversational question ("did he win it?", "what about the second one?") embeds into the wrong region of vector space and the top-k passages miss before any answer LLM runs. Query rewrite is the cheapest fix: an LLM rewrites the user's question into a **standalone, specific search query** BEFORE it is embedded, so the retrieval step sees the query the document store can actually answer.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here:

```
user question ──► inline rewrite prompt + ChatGroq ──► rewritten query ──► BGE embed ──► FAISS top-3
                                                     (raw question also embedded straight into FAISS top-3)
```

That is exactly how the shared components in `src/` work underneath: `src/retrieval/query_rewrite.py` is this same prompt + fallback loop, and `src/llms/groq.py` is a thin wrapper around the same `ChatGroq`.

The lab compares, for the same questions:

* **RAW** — plain top-k: embed the question as the user typed it;
* **REWRITTEN** — embed an LLM-rewritten version of the question instead.

Only the rewritten query reaches the inner retriever, so the vector store is untouched — the whole technique lives in the retriever layer. Questions are yes/no and short-answer items from `rag-mini-wikipedia/test.parquet` whose gold answers live inside the first 100 passages; the rewrite turns each into a standalone search query, and the gate checks that the rewritten path still surfaces the passage carrying the answer keyword.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` + `test.parquet` (3200 passages + 1000 test questions), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the Groq LLM is the *rewriter* (it never embeds); embeddings stay local BGE. Without the key the rewrite falls back to the raw question, exactly like the shared retriever's safety net.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, and `faiss-cpu`. The bootstrap cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes a deterministic head of the 3200-passage corpus (no randomness, reproducible runs); `QUESTION_IDS = [1606, 1610, 1626]` picks the vague/conversational questions whose gold answers live inside the subset — the rewrite must resolve them into standalone queries that still find the answer; `TOP_K = 3` is the retrieval depth on both paths; `LLM_MODEL` names the Groq model that does the rewriting (never the embedding); `BGE_MODEL_NAME` pins the local embedder. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
# Vague/conversational questions whose gold answers live in the subset; the
# rewrite must resolve them into standalone queries that still find the answer.
QUESTION_IDS = [1606, 1610, 1626]
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *rewriter* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages (text + ids) from `passages.parquet` — the ids are the parquet row indices, so a question's gold passage is addressable by id. `load_questions` pulls specific rows by id from `test.parquet`. `preview` flattens a passage onto one line for printing. Identical helpers to the earlier tracks keep the labs directly comparable — the only thing that changes is the retriever wrapper.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — raw vs rewritten retrieval for the same questions

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 100 passages once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. The inner retriever is a plain `similarity_search_by_vector` at `TOP_K = 3` (the inline shape of `src/retrieval/similarity.py`), and the rewrite block is the same `REWRITE_PROMPT` + `ChatGroq` the shared `QueryRewriteRetriever` wraps, including its safety net: an empty or failed LLM output falls back to the original question.

The LLM section reports **run/skip** explicitly: with `GROQ_API_KEY` in `.env` it runs (`ChatGroq`, one call per question); without the key it prints SKIP and every rewrite falls back to the raw question — the same contract the lab's `GroqLLM` follows when the key is missing.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — raw vs rewritten retrieval for the same questions
# --------------------------------------------------------------------------
REWRITE_PROMPT = """You are a search query rewriter for a RAG system.

Given the original user question, produce ONE standalone, specific search
query that would find the most relevant passages in a document store.

Rules:
- Resolve pronouns and conversational references ("it", "the second one").
- Keep the same language as the original question.
- Output only the rewritten query, nothing else.

Original question: {question}
Rewritten query:"""


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


class _ChatGroqLLM:
    """Inline stand-in for src/llms/groq.GroqLLM: ChatGroq + invoke(str) -> str."""

    def __init__(self, model: str, temperature: float = 0.0):
        self.model = model
        self.temperature = temperature
        self._llm = ChatGroq(model=model, temperature=temperature)

    def invoke(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content


class _SimilarityRetriever:
    """Inline stand-in for src/retrieval/similarity.SimilarityRetriever."""

    def __init__(self, store, embedder, top_k: int = TOP_K):
        self.store = store
        self.embedder = embedder
        self.top_k = top_k

    def retrieve(self, question: str) -> list[Document]:
        """Embed the question and return the top-k most similar documents."""
        query_embedding = self.embedder.embed_query(question)
        return self.store.similarity_search_by_vector(query_embedding, k=self.top_k)


class _QueryRewriteRetriever:
    """Inline stand-in for src/retrieval/query_rewrite.QueryRewriteRetriever."""

    def __init__(self, rewriter_llm, retriever, top_k: int = TOP_K):
        # rewriter_llm: any object exposing invoke(prompt_text) -> str.
        self.rewriter_llm = rewriter_llm
        self.retriever = retriever
        self.top_k = top_k

    def _rewrite(self, question: str) -> str:
        """Rewrite the question with the LLM, falling back to the original."""
        try:
            out = self.rewriter_llm.invoke(
                REWRITE_PROMPT.format(question=question)
            )
        except Exception:
            return question
        out = (out or "").strip()
        return out if out else question

    def retrieve(self, question: str) -> list[Document]:
        """Rewrite the question, then delegate retrieval to the inner retriever."""
        rewritten = self._rewrite(question)
        return self.retriever.retrieve(rewritten)[: self.top_k]


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs))
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = _SimilarityRetriever(store, embedder, top_k=TOP_K)
    if os.getenv("GROQ_API_KEY"):
        rewrite_llm = _ChatGroqLLM(model=LLM_MODEL)
        llm_status = f"run (ChatGroq {LLM_MODEL})"
    else:
        rewrite_llm = None
        llm_status = ("skip (no GROQ_API_KEY in the repo-root .env — "
                      "rewrites fall back to the raw question)")
        print("LLM section: SKIP —", llm_status)
    rewrite_retriever = _QueryRewriteRetriever(rewrite_llm, raw_retriever, top_k=TOP_K)

    # --- Per question: raw retrieval + the rewritten query + its retrieval --
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)
        t0 = time.perf_counter()
        rewritten = rewrite_retriever._rewrite(qtext)
        rewrite_s = time.perf_counter() - t0
        rewritten_docs = rewrite_retriever.retrieve(qtext)
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "rewritten": rewritten,
                "rewrite_s": rewrite_s,
                "raw_docs": raw_docs,
                "rewritten_docs": rewritten_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "llm_status": llm_status,
        "results": results,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/index timings and the LLM section's run/skip status; per question, the raw question, the LLM's rewritten query, and the top-1 passage of both paths; then a takeaway on why rewrite is a retriever-layer fix — the store and the embedding model never change, only the text that reaches them, at the cost of one cheap LLM call per question.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 01 — Query rewrite: fix the query before you embed it")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} rewriter")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")
    print(f"    LLM section: {exp['llm_status']}")

    print(f"\n[2] Raw vs rewritten (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      rewritten in {r['rewrite_s']:.1f}s: {r['rewritten']!r}")
        print(f"      raw      top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      rewritten top-1: {preview(r['rewritten_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    Query rewrite is a retriever-layer fix: rewrite the question")
    print("    into a standalone, specific search query, THEN embed it. The")
    print("    store and the embedding model never change — only the text")
    print("    that reaches them. It costs one cheap LLM call per question")
    print("    and pays for itself whenever users type pronouns, ellipses,")
    print("    or conversational phrasing instead of search queries.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages indexed; every question returning `TOP_K` hits on both paths; every rewrite non-empty and *different* from the raw question (the LLM resolves the vague phrasing); and the content checks — the rewritten top-3 must still carry the answer's keyword (montevideo / spanish / 1930). The keyword checks are pinned to *retrieval outcomes*, not exact LLM wording, so the gate stays stable across runs. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K rewritten hits",
                   all(len(r["rewritten_docs"]) == TOP_K for r in exp["results"])))

    # The rewrite must actually produce a standalone question — non-empty
    # and different from the original (the LLM resolves the vague phrasing).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} rewrite is non-empty",
                       bool(r["rewritten"].strip())))
        checks.append((f"{tag} rewrite differs from the raw question",
                       r["rewritten"].strip() != r["question"].strip()))

    # Content checks: the rewritten path must still surface the answer's
    # keyword. Q1606 "Is Uruguay's capital Montevideo?" -> Montevideo;
    # Q1610 "Who founded Montevideo?" -> the Spanish; Q1626 "Did Uruguay
    # host the first ever World Cup?" -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["rewritten_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} rewritten top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of embedding + 3 Groq rewrite calls on the 100-passage subset — no downloads. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three conversational questions with the rewrite the LLM produced, and the top-1 passage of the raw vs the rewritten path. Read the rewritten line like a search engineer: it is the query the *retriever* actually saw — standalone, specific, free of pronouns.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact and the LLM section ran (see the status line in the demo).


In [ ]:
verify_gate(exp)
